In [ ]:
%pip install --quiet bs4 newspaper3k lxml_html_clean python-dotenv pymysql mysql-connector-python sqlalchemy pandas

%pip install langchain_text_splitters langchain-community langchain langchain-chroma langchain-teddynote langchain-openai

In [114]:
import bs4
from langchain import hub
from langchain_community.llms import GPT4All
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.embeddings import GPT4AllEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from newspaper import Article
from bs4 import BeautifulSoup
from langchain.document_loaders import WebBaseLoader
from langchain.schema import Document
from sqlalchemy import create_engine, MetaData, Table, select, Column, Integer, String, Text
from sqlalchemy.orm import sessionmaker
from sqlalchemy.ext.declarative import declarative_base
from dotenv import load_dotenv
import os
import requests


In [115]:
load_dotenv()

CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_SCHEME = os.getenv("DB_SCHEME")

In [116]:
# collection_name = 'chroma_stock_news'
collection_name = 'chroma_stock_news_v2'


In [117]:
import bs4
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


def process_news_data_to_document(news_data):
  documents = []

  for index, row in news_data.iterrows():
    title = row['TITLE']
    content = row['CONTENT']
    
    document = Document(
        page_content=content,
        metadata={
          "title": title,        
          "stock_name": stock_name,
          "main_category": main_category,
          "sub_category": sub_category}
    )
    documents.append(document)
  
  return documents

def chunk_documents(documents, chunk_size):
    for i in range(0, len(documents), chunk_size):
      yield documents[i:i + chunk_size]

def embedding_data(documents, text_splitter):

  batch_size = 5461
  splits = text_splitter.split_documents(documents)

  for doc_batch in chunk_documents(splits, batch_size):
    Chroma.from_documents(
      documents=doc_batch,
      embedding=OpenAIEmbeddings(model='text-embedding-3-large'),
      collection_name='chroma_stock_news',
      persist_directory="./chroma_3-large"
    )

In [118]:

def search_naver_news(query, display=100, start=1, sort='date'):
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET
    }
    params = {
        "query": query,  # 검색어
        "display": display,  # 가져올 결과 수
        "start": start,  # 검색 시작 위치
        "sort": sort  # 정렬 기준: date(날짜순), sim(유사도순)
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        return response.json() 
    else:
        print("Error:", response.status_code)
        return None

In [ ]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("RAG")

In [120]:
engine = create_engine(f'mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}/{DB_SCHEME}')

Session = sessionmaker(bind=engine)
session = Session()

metadata = MetaData()

# 테이블 정의
stock_table = Table('STOCK', metadata, autoload_with=engine)
sub_category_table = Table('SUB_CATEGORY', metadata, autoload_with=engine)
main_category_table = Table('MAIN_CATEGORY', metadata, autoload_with=engine)

# 쿼리 작성
stmt = select(
    stock_table.c.STOCK_ID,
    stock_table.c.STOCK_NAME,
    sub_category_table.c.SUB_CATEGORY_NAME,
    main_category_table.c.MAIN_CATEGORY_NAME
).join(
    sub_category_table, stock_table.c.SUB_CATEGORY_ID == sub_category_table.c.SUB_CATEGORY_ID
).join(
    main_category_table, sub_category_table.c.MAIN_CATEGORY_ID == main_category_table.c.MAIN_CATEGORY_ID
)

# 쿼리 실행
result = session.execute(stmt)

# 결과를 리스트로 변환
stock_list = [(row[0], row[1], row[2], row[3]) for row in result]

# 세션 종료
session.close()

# 결과 출력
print(stock_list)


[(28244, '한국테크놀로지', '4차 산업', '숙박 및 음식'), (29327, '동아타이어', '4차 산업', '숙박 및 음식'), (10763, 'Test Stock', 'IT기기', '숙박 및 음식'), (27524, '서부T&D', '숙박·음식', '숙박 및 음식'), (27880, '아난티', '숙박·음식', '숙박 및 음식'), (29126, '디딤이앤에프', '숙박·음식', '숙박 및 음식'), (27325, '한탑', '음식료·담배', '숙박 및 음식'), (27365, '대주산업', '음식료·담배', '숙박 및 음식'), (27428, '창해에탄올', '음식료·담배', '숙박 및 음식'), (27474, '푸드웰', '음식료·담배', '숙박 및 음식'), (27488, '한일사료', '음식료·담배', '숙박 및 음식'), (27498, '매일홀딩스', '음식료·담배', '숙박 및 음식'), (27616, '엠에스씨', '음식료·담배', '숙박 및 음식'), (27756, '현대사료', '음식료·담배', '숙박 및 음식'), (27779, '진로발효', '음식료·담배', '숙박 및 음식'), (27842, '풍국주정', '음식료·담배', '숙박 및 음식'), (27876, '케이씨피드', '음식료·담배', '숙박 및 음식'), (27892, '팜스토리', '음식료·담배', '숙박 및 음식'), (28001, '이지홀딩스', '음식료·담배', '숙박 및 음식'), (28126, '국순당', '음식료·담배', '숙박 및 음식'), (28367, '체리부로', '음식료·담배', '숙박 및 음식'), (28452, '우리손에프앤지', '음식료·담배', '숙박 및 음식'), (28519, '우리바이오', '음식료·담배', '숙박 및 음식'), (28578, '동우팜투테이블', '음식료·담배', '숙박 및 음식'), (28616, '아미코젠', '음식료·담배', '숙박 및 음식'), (28728, '우양', '음식료·담배', '숙박 및 음식'), (

In [121]:
Base = declarative_base()

class News(Base):
    __tablename__ = 'NEWS'
    news_id = Column(Integer, primary_key=True, autoincrement=True)
    stock_id = Column(Integer, nullable=False)
    title = Column(String(20), nullable=True)
    content = Column(Text, nullable=True)
    link = Column(String(100), nullable=True)
    image = Column(String(100), nullable=True)

engine = create_engine(f'mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}/{DB_SCHEME}')

Session = sessionmaker(bind=engine)
session = Session()

metadata = MetaData()

def save_to_database(news_data):
    try:
        for news_item in news_data:
            data = News(
                stock_id=news_item['stock_id'],
                title=news_item['title'],
                link=news_item['link'],
                content=news_item['content'],
                image=news_item['image']
            )
            session.add(data)

        session.commit()
    except Exception as e:
        session.rollback() 
        print(f"DB error occurred: {e}")


C:\Users\student\AppData\Local\Temp\ipykernel_18884\1720991271.py:1: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [122]:

def collect_news_data(news_data_list):
  result_list = []
  docs = []
  for news_item in news_data_list:
    stock_id = news_item['stock_id']
    sub_category_name = news_item['sub_category_name']
    main_category_name =news_item['main_category_name'] 
    for news_url in news_item['news_url_list']:
      if (news_url.startswith("https://n.news.naver.com") == False):
        continue
      response = requests.get(news_url)
      html_content = response.content

      soup = BeautifulSoup(html_content, 'html.parser')

      title = soup.find("div", class_="media_end_head_title").get_text(strip=True)
      content = soup.find("div", class_="newsct_article _article_body").get_text(strip=True)
      # publish_date = soup.find("span", class_="media_end_head_info_datestamp_time").get_text(strip=True)
      contents_div = soup.find("div", id="contents")
      image_tag = contents_div.find("img")
      if image_tag:
        # 먼저 'src' 속성을 확인하고 없으면 'data-src' 속성을 확인
        image_src = image_tag.get('src') or image_tag.get('data-src')
      # publisher = soup.find("img", class_="media_end_head_top_logo_img")['title']

      result_list.append({
        'stock_id': stock_id,
        'title': title,
        'content': content,
        'link': news_url,
        # 'publish_date': publish_date,
        'image': image_src
      })

      document = Document(
        page_content=content,
        metadata={"title": title, "sub_category" : sub_category_name, "main_category" : main_category_name}
      )
      docs.append(document)

  return {"result_list": result_list, "docs": docs}
    # else:
    #   article = Article(news_url, language = 'ko') 

    #   article.download()
    #   article.parse()
    #   title = article.title
    #   text = article.text
    #   date = article.publish_date
    #   image_url = article.top_image

    #   doc_content = title + "\n" + text
    #   doc = Document(page_content=doc_content)
    #   docs.append(doc)

In [124]:

def news_test(stock_list):
  news_data_list = []
  result_list = []
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

  for stock_id, stock_name, sub_category_name, main_category_name in stock_list:
    result = search_naver_news(stock_name)
    news_url_list = []
    if result:
      for idx, item in enumerate(result['items']):
        news_url_list.append(item['link'])

    news_data_list.append({
      'stock_id' : stock_id,
      'sub_category_name' : sub_category_name,
      'main_category_name' : main_category_name,
      'news_url_list' : news_url_list
    })

    if(len(news_data_list) >= 30):
      news_data_docs_and_list = collect_news_data(news_data_list)
      result_list = news_data_docs_and_list['result_list']
      docs = news_data_docs_and_list['docs']
      print(f"{len(result_list)} jobs processing...")
      # save_to_database(result_list)
      embedding_data(docs, text_splitter)
      print("save success")
      result_list = []
      docs = []
      news_data_list=[]



news_test(stock_list)

519 jobs processing...
save success
581 jobs processing...
save success
685 jobs processing...
save success
604 jobs processing...
save success
619 jobs processing...
save success
551 jobs processing...
save success
944 jobs processing...
save success
608 jobs processing...
save success
630 jobs processing...
save success
725 jobs processing...
save success
768 jobs processing...
save success
742 jobs processing...
save success
377 jobs processing...
save success
589 jobs processing...
save success
691 jobs processing...
save success
686 jobs processing...
save success
526 jobs processing...
save success
689 jobs processing...
save success
481 jobs processing...
save success
429 jobs processing...
save success
414 jobs processing...
save success
439 jobs processing...
save success
427 jobs processing...
save success
460 jobs processing...
save success
543 jobs processing...
save success
390 jobs processing...
save success
473 jobs processing...
save success
426 jobs processing...
save 

AttributeError: 'NoneType' object has no attribute 'get_text'

In [ ]:
news_data_list = []

for stock_id, stock_name in stock_list:
  result = search_naver_news(stock_name)

  news_url_list = []
  all_splits = []

  if result:
    for idx, item in enumerate(result['items']):
      news_url_list.append(item['link'])

  news_data_list.append({
    'stock_id' : stock_id,
    'news_url_list' : news_url_list
  })
  
  if(len(news_data_list) >= 50):
    print("now saving...")
    result_list = []
    collect_news_data(news_data_list, all_splits, result_list)
    save_to_database(result_list)
    print(f'{len(result_list)} news saved')
    news_data_list = []


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

for news_url in news_data:
    article = Article(news_url, language = 'ko') 

    article.download()
    print(news_url)
    article.parse()
    title = article.title
    text = article.text
    date = article.publish_date
    image_url = article.top_image

    doc_content = title + "\n" + text
    doc = Document(page_content=doc_content)
    docs = [doc] 

    # add_to_verctor_db(text_splitter, docs)
    print(f"URL: {news_url}\n {title}\n {text}\n {date}\n{image_url}\n")

In [ ]:
all_splits = []


for news_url in news_data:
    if news_url.startswith("https://n.news.naver.com"):
        # 각 뉴스 URL에 대해 WebBaseLoader를 설정합니다.
        loader = WebBaseLoader(
            web_paths=(news_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    "div",
                    attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
                )
            ),
        )
        article = Article(news_url, language = 'ko') 

        article.download()
        article.parse()

        image_url = article.top_image
        docs = loader.load()
        print('docs :{docs}\n image_url :{image_url}')
    else:

        article = Article(news_url, language = 'ko') 

        article.download()
        article.parse()
        title = article.title
        text = article.text
        date = article.publish_date
        image_url = article.top_image

        doc_content = title + "\n" + text
        doc = Document(page_content=doc_content)
        docs = [doc] 

    print(f"URL: {news_url} {image_url}")
    
    splits = text_splitter.split_documents(docs)
    all_splits.extend(splits)


## OPENAI 모델

In [24]:
import pandas as pd
from sqlalchemy import create_engine

def fetchDBNews():

  # SQLAlchemy 엔진 생성 (MySQL 연결 문자열)
  engine = create_engine(f'mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}/{DB_SCHEME}')

  # SQL 쿼리 실행하여 title과 content 가져오기
  query = "SELECT TITLE, CONTENT FROM NEWS"
  news_data = pd.read_sql(query, engine)

  # 데이터 미리 보기 (첫 5개의 데이터)
  return news_data

In [138]:

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

# 환경변수를 불러옴
load_dotenv()

llm = ChatOpenAI(model='gpt-4o')
# news_data = fetchDBNews()

# OpenAI에서 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = OpenAIEmbeddings(model='text-embedding-3-large')


vectorstore = Chroma(collection_name='chroma_stock_news', persist_directory="./chroma_3-large", embedding_function=embedding)
retriever = vectorstore.as_retriever()

# system_prompt = (
# """당신은 최신 동향을 분석하고 제공하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context)을 바탕으로 주식 시장에 대한 동향을 전달하는 것입니다.
# 검색된 다음 문맥(context)을 사용하여 최신 주식 시장 동향을 요약해 주세요. 만약, 주어진 문맥(context)에서 동향을 찾을 수 없다면, `주어진 정보에서 시장 동향에 대한 정보를 찾을 수 없습니다`라고 답하세요.
# 한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

# #문맥(Context): 
# {context} 

# #시장 동향 요약:"""
# )

system_prompt = (
  """당신은 주식 종목을 분석하고 제공하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context)을 바탕으로 주식 종목에 대한 동향을 전달하는 것입니다.
검색된 다음 문맥(context)을 사용하여 최신 주식 종목을 요약해 주세요. 만약, 주어진 문맥(context)에서 동향을 찾을 수 없다면, `주어진 정보에서 주식 종목에 대한 정보를 찾을 수 없습니다`라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#문맥(Context): 
{context} 

#시장 동향 요약:"""
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [155]:
trend_result = rag_chain.invoke({"input" : "두산 종목 동향"})

In [ ]:
trend_result

In [134]:
trend_system_prompt = (
"""당신은 최신 금융 동향을 분석하고 제공하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context)을 바탕으로 금융 시장에 대한 동향을 전달하는 것입니다.
검색된 다음 문맥(context)을 사용하여 금융 시장 동향을 요약해 주세요. 메타데이터에 포함된 `main_category`를 기반으로 관련 내용을 전달해야 합니다.
만약, 주어진 문맥(context)에서 동향을 찾을 수 없다면, `주어진 정보에서 금융 시장 동향에 대한 정보를 찾을 수 없습니다`라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

메타데이터:
- 서브 카테고리 (main_category): {main_category}

#문맥(Context): 
{context} 

#금융 시장 동향 요약:"""
)

# 프롬프트에 필요한 요소를 준비하여 생성
trend_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", trend_system_prompt),
        ("human", "{input}"),
    ]
)
trend_question_answer_chain = create_stuff_documents_chain(llm, trend_prompt)
trend_rag_chain = create_retrieval_chain(retriever, trend_question_answer_chain)
stocK_trend = trend_rag_chain.invoke({"input": "입력된 메인 카테고리에 대한 동향", "main_category": "숙박 및 음식'에 대한 동향"})

In [ ]:
stocK_trend

In [21]:
from langchain_teddynote.messages import stream_response

In [ ]:
from langchain_core.prompts import PromptTemplate

# prompt = PromptTemplate.from_template(
#     """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
# 검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
# 한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

# #Question: 
# {question} 

# #Context: 
# {context} 

# #Answer:"""
# )

prompt = PromptTemplate.from_template(
    """당신은 최신 금융 동향을 분석하고 제공하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context)을 바탕으로 금융 시장에 대한 동향을 전달하는 것입니다.
검색된 다음 문맥(context)을 사용하여 최신 금융 시장 동향을 요약해 주세요. 만약, 주어진 문맥(context)에서 동향을 찾을 수 없다면, `주어진 정보에서 금융 시장 동향에 대한 정보를 찾을 수 없습니다`라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#문맥(Context): 
{context} 

#금융 시장 동향 요약:"""
)

In [4]:
import pandas as pd
from sqlalchemy import create_engine

def fetchNews():
  engine = create_engine(f'mysql+pymysql://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}/{DB_SCHEME}')

  query = "SELECT TITLE, CONTENT, SUB_CATEGORY FROM NEWS"
  news_data = pd.read_sql(query, engine)

  return news_data


In [5]:
import bs4
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

newsData = fetchNews()
documents = []

for index, row in newsData.iterrows():
    title = row['TITLE']
    content = row['CONTENT']
    
    document = Document(
        page_content=content,
        metadata={"title": title}
    )
    documents.append(document)

splits = text_splitter.split_documents(documents)
def chunk_documents(documents, chunk_size):
    for i in range(0, len(documents), chunk_size):
        yield documents[i:i + chunk_size]

# 최대 배치 크기를 5461로 설정
batch_size = 5461
splits = text_splitter.split_documents(documents)  # 문서 분리

# 각 배치별로 Chroma에 저장
for doc_batch in chunk_documents(splits, batch_size):
    vectorstore = Chroma.from_documents(
        documents=doc_batch,
        embedding=OpenAIEmbeddings(),
        collection_name='chroma_stock_news',
        persist_directory="./chroma"
    )

In [8]:

vectorstore = Chroma(collection_name='chroma_stock_news', persist_directory="./chroma", embedding_function=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

system_prompt = (
"""당신은 최신 금융 동향을 분석하고 제공하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context)을 바탕으로 금융 시장에 대한 동향을 전달하는 것입니다.
검색된 다음 문맥(context)을 사용하여 최신 금융 시장 동향을 요약해 주세요. 만약, 주어진 문맥(context)에서 동향을 찾을 수 없다면, `주어진 정보에서 금융 시장 동향에 대한 정보를 찾을 수 없습니다`라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#문맥(Context): 
{context} 

#금융 시장 동향 요약:"""
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [9]:
result = rag_chain.invoke({"input" : "한국 금융 시장 동향"})

In [ ]:
result